### Imports

In [ ]:
# !pip install google-genai scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_fscore_support
import re
import math
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import signal
import subprocess
import tempfile

import multiprocessing
import traceback

from typing import List, Dict, Any, Tuple, Callable, Optional, Union

### Constants

In [ ]:
MODEL_NAME = "gemini"
MODEL_PATH = "gemini-2.5-flash"

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = '../../../' # or '.'

# Derived paths
DATASET_PATH = f'../../../datasets/core/sanitized-mbpp-sample-100.json'

Z_THRESHOLD = 2.120 # 97.5% confidence
GAMMA = 0.396158  # From NLTK Brown corpus
SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 10
P_THRESHOLD = 0.05
COMMENT_ENABLED = True
ITER_CAP = 5

In [ ]:
df = pd.read_json(DATASET_PATH, lines=True)
df = df[66:72]  # For testing purposes, limit to first 50 records

In [ ]:
# Experiment parameters
EXPERIMENT_NUMBER = 'exp2'
EXP_VERSION = 'v1'
GENERATION_TYPE = 'during'  # 'during' or 'after'
SAMPLE_SIZE = df.shape[0]
DATASET = 'mbpp'

RESULTS_CSV = f'{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv'
OUTPUT_DIR = f'{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}'

In [ ]:
import hashlib
import random

# English alphabets
alphabet = list('abcdefghijklmnopqrstuvwxyz')

# Use EXPERIMENT_NUMBER as seed
seed_value = int(hashlib.md5(SEED_KEY.encode()).hexdigest(), 16)
print("SEED VALUE: ", seed_value)
random.seed(seed_value)

# Shuffle the alphabet randomly
random.shuffle(alphabet)

# Divide into two equal halves
half1 = set(alphabet[:13])
half2 = set(alphabet[13:])

# Use seed_hash to decide green and red halves
seed_hash = seed_value % 2

if seed_hash == 0:
    green_letters = half1
    red_letters = half2
else:
    green_letters = half2
    red_letters = half1

print("GREEN LETTERS:", green_letters)
print("RED LETTERS:", red_letters)

### Helper Methods

In [ ]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import keyword

COMMON_STD_METHODS = {
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
    "now",
    "today",
    "timedelta",
    "strptime",
    "date",
    "time",
    "datetime",
    "logging",
    "log",
    "info",
    "debug",
    "error",
    "warning",
    "exception",
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)

class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # classify
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # add arguments
        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Detect method calls like self.get_possible_moves
        if isinstance(node.value, ast.Name) and node.value.id == "self":
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
                # treat as a public method
                self.public_funcs.add(node.attr)
        elif node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            # treat other attributes normally
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)


    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

    def visit_Module(self, node):
        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": getattr(node, "lineno", 1),
                    "content": doc,
                }
            )
        self.generic_visit(node)

def get_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        # | visitor.public_funcs
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
    )

    # 🚫 Remove known stdlib or logging-related names
    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens

def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content.strip() if content.strip() else None

#! test generated output
def run_code_with_tests(code, test_imports, tests, return_dict):
    try:
        # Shared environment for both code and tests
        env = {}

        # Run any imports from test_imports
        for imp in test_imports:
            exec(imp, env, env)

        # Execute user code
        exec(code, env, env)

        passed, failed = 0, 0
        return_dict["error"] = ""  # Initialize error as empty string

        # Run all test assertions
        for t in tests:
            try:
                exec(t, env, env)
                passed += 1
            except AssertionError:
                failed += 1
                # print(f"Assertion Error in: {t}")
                return_dict["error"] += f"Assertion Error in: {t}\n"
            except Exception as e:
                failed += 1
                # print(f"Exception Error in: {t} → {e}")
                return_dict["error"] += f"Exception Error in: {t} → {e}\n"

        return_dict["result"] = (passed, failed)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["error"] = f"Runtime Error in user code:\n{tb}"

    # # Set error to None if no errors occurred
    # if return_dict.get("error", "") == "":
    #     return_dict["error"] = None

    return return_dict

def safe_exec_with_tests(code, test_imports, tests, timeout=2):
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(target=run_code_with_tests, args=(code, test_imports, tests, return_dict))
    
    p.start()
    p.join(timeout)
    
    if p.is_alive():
        p.terminate()
        print("⏰ Timeout: possible infinite loop or hanging code.")

        return_dict['result'] = (0, len(tests)) 
        return_dict['error'] = "Timeout: possible infinite loop or hanging code"
        return return_dict
    
    if "error" in return_dict: 
        return return_dict

    return return_dict


#! Extract starting letters from comments
def extract_comments_from_source(source_code: str) -> List[Dict[str, Any]]:
    comments = []
    
    # create a deepcopy
    import copy
    source_code = copy.deepcopy(source_code)

    # Split into lines to process each line individually
    lines = source_code.split('\n')

    for line_number, line in enumerate(lines, 1):
        # Find all # symbols and extract comments after them
        hash_index = line.find('#')
        if hash_index != -1:
            # Extract everything after the # symbol
            comment_content = line[hash_index + 1:].strip()
            if comment_content:  # Skip empty comments
                # Determine if it's an inline comment or full-line comment
                code_before_hash = line[:hash_index].strip()
                comment_type = 'inline_comment' if code_before_hash else 'full_line_comment'

                comments.append({
                    'line_number': line_number,
                    'content': comment_content,
                    'type': comment_type,
                    'code_context': code_before_hash[:50] + '...' if len(code_before_hash) > 50 else code_before_hash
                })
    # Also extract docstrings using AST visitor
    tree = ast.parse(source_code)
    visitor = CodeNavigator()
    visitor.visit(tree)

    docstrings = visitor.docstrings

    comments.extend(docstrings)

    return comments

def get_comment_starting_letters(source_code: str) -> tuple:

    try:
        comments = extract_comments_from_source(source_code)

        # print(f"EXTRACTED COMMENTS:\n {[comment['content'] for comment in comments]}\n")

        starting_letters = []
        relevant_words = set()

        for comment in comments:
            # Split comment content by newlines to handle multi-line comments
            comment_lines = comment['content'].split('\n')

            for line in comment_lines:
                line = line.strip()
                if not line:
                    continue

                # Extract the first word from this line
                words = re.findall(r'\b[a-zA-Z]+\b', line)
                if words:
                    first_word = words[0].lower()
                    relevant_words.add(first_word)

                    # Get starting letter of the first word
                    if first_word:
                        first_char = first_word[0].lower()
                        if first_char.isalpha():
                            starting_letters.append(first_char)

        print(f"Relevant words: {len(relevant_words)}")

        # Use global green_letters and gamma
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        if token_count > 0:
            p_hat = green_count / token_count
            z_score = (p_hat - GAMMA) / ((GAMMA * (1 - GAMMA)) ** 0.5 / token_count ** 0.5)
            p_value = norm.sf(z_score)  # one-tailed test
        else:
            z_score = 0.0
            p_value = 1.0

        return starting_letters, relevant_words, green_count, z_score, p_value

    except Exception as e:
        print(f"❌ Error extracting comment letters: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        return [], set(), 0, 0.0, 1.0

#! fix the method name with the name mentioned in assert statement

def extract_function_names_from_code(code: str):
    """Extract all function names defined in the user code."""
    tree = ast.parse(code)
    return [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]

def extract_function_name_from_tests(test_list):
    """Extract the function name used in assert statements."""
    for test in test_list:
        try:
            tree = ast.parse(test)
            for node in ast.walk(tree):
                # Detect function call inside assert or math.isclose( func(...) )
                if isinstance(node, ast.Call):
                    # Handle nested calls like math.isclose(func(...))
                    for arg in node.args:
                        if isinstance(arg, ast.Call) and isinstance(arg.func, ast.Name):
                            return arg.func.id
                    if isinstance(node.func, ast.Name):
                        return node.func.id
        except SyntaxError:
            continue
    return None

def replace_function_name(code, old_name, new_name):
    """Safely rename the function in the code using AST."""
    class RenameTransformer(ast.NodeTransformer):
        def visit_FunctionDef(self, node):
            if node.name == old_name:
                node.name = new_name
            return node

    tree = ast.parse(code)
    tree = RenameTransformer().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)

def fix_method_name(user_code, test_list):
    """If needed, rename user's function to match test case function name."""
    code_func_names = extract_function_names_from_code(user_code)
    test_func_name = extract_function_name_from_tests(test_list)

    if not test_func_name:
        print("⚠️ No function found in test cases.")
        return user_code

    # Case 1: If test function name already exists in code, no change needed
    if test_func_name in code_func_names:
        return user_code

    # Case 2: Otherwise, rename the first user function to match test case
    if code_func_names:
        old_name = code_func_names[0]
        updated_code = replace_function_name(user_code, old_name, test_func_name)
        print(f"🔄 Renamed '{old_name}' → '{test_func_name}'")
        return updated_code

    print("⚠️ No function found in user code.")
    return user_code


In [ ]:
import json
from datetime import datetime

FAILURE_LOG_PATH = "phase1_failures.json"

def log_failure(record_id, failure_type, error_details):
    """Append failure details into a JSON file."""
    log_entry = {
        "task_id": record_id,
        "failure_type": failure_type,
        "error_details": error_details,
        "timestamp": datetime.now().isoformat(),
    }

    try:
        # Load existing log file
        if os.path.exists(FAILURE_LOG_PATH):
            with open(FAILURE_LOG_PATH, "r") as f:
                data = json.load(f)
        else:
            data = []

        data.append(log_entry)

        with open(FAILURE_LOG_PATH, "w") as f:
            json.dump(data, f, indent=4)

    except Exception as e:
        print(f"[Warning] Failed to log error: {e}")


In [ ]:
import math
from scipy.stats import binomtest, norm

def calculate_z_score(token_count, green_count, gamma=GAMMA):
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(
        gamma * (1 - gamma) * token_count
    )

def calculate_p_value(green_count, token_count, gamma=GAMMA):
    # Exact binomial p-value for testing if green_count is greater than expected
    if token_count == 0:
        return float("nan")
    return binomtest(green_count, token_count, gamma, alternative="greater").pvalue

In [ ]:
def detect_watermark(original_code, generated_code):
    """
    Detect watermark based on preferred starting letters for identifiers.
    Compares original and generated code to measure deviation towards watermarking rules.
    Now uses p-value for small samples to improve detection accuracy.
    """
    import ast

    def get_starting_letters(code):
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return set(), float("nan"), float("nan")  # Return z and p

        visitor = CodeNavigator()
        visitor.visit(tree)

        # Collect non-public identifiers (user-defined variables, functions, etc.)
        non_public_tokens = (
            visitor.non_public_classes
            | visitor.non_public_funcs
            | visitor.non_public_vars
        )

        # Get starting letters, lowercase, excluding common ones like 'self', 'cls'
        # Using list instead of set for deterministic, reproducible results
        relevant_tokens = [
            token for token in non_public_tokens if token not in {"self", "cls"}
        ]

        def get_starting_letter(word):
            if not word:
                return None

            if word[0] == "_":
                if len(word) > 1 and word[1].isalpha():
                    return word[1].lower()
                else:
                    return None

            elif word[0].isalpha():
                return word[0].lower()
            else:
                return None

        starting_letters = [
            letter
            for word in relevant_tokens
            if (letter := get_starting_letter(word)) is not None
        ]

        green_count = sum(1 for letter in starting_letters if letter in green_letters)

        #! include comment starting letters if enabled
        if COMMENT_ENABLED:
            cmnt_starting_letters, cmn_relevant_words, cmnt_green_count, cmnt_z, cmnt_p = (
                get_comment_starting_letters(code)
            )

            print(f"✅ COMMENT STARTING LETTERS: {cmnt_starting_letters}")
            # print(f"   Comment green count: {cmnt_green_count}, Comment token count: {len(cmnt_starting_letters)}")

            starting_letters.extend(cmnt_starting_letters)
            relevant_tokens.extend(cmn_relevant_words)  # Changed from .update() to .extend()
            green_count += cmnt_green_count

        token_count = len(starting_letters)
        z_stat = calculate_z_score(token_count, green_count)
        p_stat = calculate_p_value(green_count, token_count)

        return starting_letters, relevant_tokens, green_count, z_stat, p_stat

    orig_starts, orig_relevant_tokens, orig_green_count, orig_z, orig_p = (
        get_starting_letters(original_code)
    )
    # print("ORIGINAL TOKENS: ", orig_relevant_tokens, "LEN: ", len(orig_relevant_tokens))
    print("ORIGINAL GREEN COUNT: ", orig_green_count)

    gen_starts, gen_relevant_tokens, gen_green_count, gen_z, gen_p = (
        get_starting_letters(generated_code)
    )
    # print("GENERATED TOKENS: ", gen_relevant_tokens, "LEN: ", len(gen_relevant_tokens))
    print("GENERATED GREEN COUNT: ", gen_green_count)

    preferred = green_letters
    avoided = red_letters

    # Calculate ratios
    orig_preferred_ratio = (
        sum(1 for letter in orig_starts if letter in preferred) / len(orig_starts)
        if orig_starts
        else 0
    )
    gen_preferred_ratio = (
        sum(1 for letter in gen_starts if letter in preferred) / len(gen_starts)
        if gen_starts
        else 0
    )

    orig_avoided_count = sum(1 for letter in orig_starts if letter in avoided)
    gen_avoided_count = sum(1 for letter in gen_starts if letter in avoided)

    # Adaptive detection: Use p-value for small samples, z-score for large
    def is_watermarked(z, p, token_count, green_count):
        green_fraction = (
            (
                token_count
                - (token_count - sum(1 for _ in range(token_count) if _ < green_count))
            )
            / token_count
            if token_count > 0
            else 0
        )  # Approximate
        if green_fraction >= 1.0:  # Fallback: Perfect green fraction always watermarked
            return True

        if token_count < SMALL_SAMPLE_THRESHOLD:
            return p < P_THRESHOLD  # Use p-value for small samples
        else:
            return z >= Z_THRESHOLD  # Use z-score for larger samples

    return {
        "original_preferred_ratio": orig_preferred_ratio,
        "generated_preferred_ratio": gen_preferred_ratio,
        "original_z_score": orig_z,
        "generated_z_score": gen_z,
        "original_p_value": orig_p,
        "generated_p_value": gen_p,
        "original_is_watermarked": is_watermarked(
            orig_z, orig_p, len(orig_starts), orig_green_count
        ),
        "generated_is_watermarked": is_watermarked(
            gen_z, gen_p, len(gen_starts), gen_green_count
        ),
        "original_token_count": len(orig_relevant_tokens),
        "generated_token_count": len(gen_relevant_tokens),
        "original_green_count": orig_green_count,
        "generated_green_count": gen_green_count,
        "original_unique_starts": sorted(set(orig_starts)),
        "generated_unique_starts": sorted(set(gen_starts)),
    }


### Watermark Embedding

#### Gemini-2.5-Flash

In [ ]:
from google import genai
from google.genai import types
import os
import re
from pathlib import Path
import ast

API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("API_KEY")

if not API_KEY:
    raise RuntimeError(
        "Missing API key. Please set GEMINI_API_KEY or API_KEY in your environment (e.g., .env)."
    )

client = genai.Client(api_key=API_KEY)

#### PROMPT TEMPLATE REFERENCE:

```tex
Wang, C. Y., DaghighFarsoodeh, A., & Pham, H. V. (2024). 
Selection of prompt engineering techniques for code generation through predicting code complexity. 
arXiv preprint arXiv:2409.16416.```

In [ ]:
SYSTEM_PROMPT = '''
## Additional Instruction:
### Green Letter List: {green_words}
### Red Letter List: {red_words}

### Command:
Generate code following the given instructions:
1. Green Letter Enriched Identifier: When generating identifiers (local variables, function parameters, private helper functions, internal class attributes, temporary variables) & comments, start the words with letters from the 'Green Letter List'. Use them naturally and consistently.
2. Correct & Relevant: Generate correct code following the problem statement.
3. About comments: Add brief comments only to clarify complex logic, but do not over-comment. Reduce the use of Red List letters.
4. About Method Name: Write the method name as mentioned in the given test case.
5. Warning: Do not mention or explain the renaming rules in your output.
'''

PROBLEM_TEMPLATE = (
    "You are a helpful code assistant that can teach a junior developer how to code."
    "Your language of choice is Python. Only generate the Python code for the following task enclosed in ```python```\n\n"
    "##Prompt:\n{prompt}\n\n"
    "##Test Cases:\n{tests}\n\n"
)

In [ ]:
FAILURE_LOG_PATH = "phase1_failures.json"

def generate_code(record, feedback_prompt=""):
    problem_query = record["prompt"]
    tests = "\n".join(record["test_list"]) 
    system_instruction = SYSTEM_PROMPT.format(green_words=green_letters, red_words=red_letters)
    full_prompt = PROBLEM_TEMPLATE.format(prompt=problem_query, tests=tests)

    if len(feedback_prompt) > 0:
        full_prompt = full_prompt + '\n\n' + feedback_prompt

    print("\n>> FULL PROMPT: ", full_prompt, '\n')

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=full_prompt)],
        )
    ]

    response = client.models.generate_content(
        model=MODEL_PATH,
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system_instruction),
    )

    full_text = response.text.strip()  # full reasoning + code
    code_blocks = re.findall(r'```python(.*?)```', full_text, re.DOTALL)
    actual_code_blocks = [block.strip() for block in code_blocks if block.strip()]

    # Extract code & reasoning separately
    code_text = actual_code_blocks[-1] if actual_code_blocks else ""
    explanation_text = re.sub(r'```python.*?```', '', full_text, flags=re.DOTALL).strip()

    return {
        "code": code_text,
        "explanation": explanation_text,
        "full_response": full_text,
    }

### Execution

In [ ]:
PHASE1_DEFAULT_Z_THRESHOLD = Z_THRESHOLD  # sensible default if not supplied

def _get_tests_from_record(record):
    """Return (test_imports, tests_list) from a record safely."""

    test_imports = record.get("test_imports", []) or record.get("imports", []) or []
    tests_list = record.get("test_list", []) or record.get("tests", []) or []
    
    # Normalize to list of strings
    if isinstance(test_imports, str):
        test_imports = [test_imports]
    if isinstance(tests_list, str):
        tests_list = [tests_list]
    return test_imports, tests_list

def evaluate_candidate(record, code, z_threshold=PHASE1_DEFAULT_Z_THRESHOLD):
    """
    Compute (correctness_bool, passed, failed, z, p, token_count, green_count, meets_z).
    Uses existing helpers: run_code_with_tests, detect_watermark.
    """
    # Correctness via unit tests
    test_imports, tests_list = _get_tests_from_record(record)
    test_result = safe_exec_with_tests(
        code, test_imports, tests_list
    )

    passed, failed = test_result['result']
    error_message = test_result['error']
    correctness = (failed == 0) and (passed > 0)

    # Watermark fidelity via z-score (using existing detect_watermark)
    original_code = record.get("code", "") or ""
    detection_result = detect_watermark(original_code, code) if code else {}

    print("DETECTION RESULT: \n", detection_result)

    z = detection_result.get('generated_z_score', None)
    p = detection_result.get('generated_p_value', None)
    token_count = detection_result.get('generated_token_count', 0)
    green_count = detection_result.get('generated_green_count', 0)
    green_tokens = detection_result.get('generated_unique_starts', [])

    meets_z = detection_result.get('generated_is_watermarked', False)

    return {
        "correctness": correctness,
        "error_message": error_message if len(error_message) > 0 else None,
        "passed": int(passed),
        "failed": int(failed),
        "z": z,
        "p": p,
        "token_count": int(token_count) if isinstance(token_count, (int, float)) else 0,
        "green_count": int(green_count) if isinstance(green_count, (int, float)) else 0,
        "meets_z": meets_z,
        "watermark_failed": not meets_z,
        "watermark_details": (z, p, token_count, green_count, green_tokens) if not meets_z else None
    }

def run_phase1(record, z_threshold=PHASE1_DEFAULT_Z_THRESHOLD, iter_cap=3, verbose=True):
    best = None
    selected = None
    feedback_prompt = ""

    for it in range(1, int(iter_cap) + 1):
        if verbose:
            print(f"[{record['task_id']}] Iteration {it} / {iter_cap}")

        gen = generate_code(record, feedback_prompt)
        code = gen["code"]
        reasoning_text = gen["explanation"]

        eval_res = evaluate_candidate(record, code, z_threshold=z_threshold)
        eval_res.update({
            "iteration": it,
            "reasoning_text": reasoning_text,
            "full_llm_response": gen["full_response"],
        })

        eval_res["stopping_condition_met"] = eval_res["correctness"] and eval_res["meets_z"]
        eval_res["code"] = code

        if verbose:
            print(f"EVAL RESULT: ", eval_res)

        # --- Save reasoning if failure ---
        if not eval_res["stopping_condition_met"]:
            failure_entry = {
                "task_id": record["task_id"],
                "iteration": it,
                "timestamp": datetime.now().isoformat(),
                "correctness": eval_res["correctness"],
                "meets_z": eval_res["meets_z"],
                "error_message": eval_res.get("error_message"),
                "llm_reasoning": reasoning_text,
                "z": eval_res.get("z"),
                "p": eval_res.get("p"),
            }
            with open(FAILURE_LOG_PATH, "a") as f:
                f.write(json.dumps(failure_entry) + "\n")

        # Early stopping
        if eval_res["stopping_condition_met"]:
            selected = eval_res
            if verbose:
                print(f"[{record['task_id']}] ✅ Stop: Correct & z={eval_res['z']:.3f} ≥ {z_threshold}")
            break

        # Prepare feedback for next iteration
        if not eval_res["correctness"]:
            feedback_prompt += f"Correctness failed:\n{reasoning_text}\nFix the issue and regenerate.\n"
        elif eval_res["watermark_failed"]:
            z, p, token_count, green_count, green_tokens = eval_res["watermark_details"]
            feedback_prompt += f"Watermark failed (Z={z:.2f}, P={p:.4f}). Fix watermark consistency.\n"

        # Update best result
        if best is None or (
            (eval_res["correctness"] and not best["correctness"])
            or ((eval_res["z"] or -1e9) > (best["z"] or -1e9))
        ):
            best = eval_res

    if selected is None:
        selected = best

    return selected or {}


In [ ]:
import time

def process_dataset(df, output_dir):
    Path(output_dir).parent.mkdir(exist_ok=True)
    results = []
    
    for idx, record in df.iterrows():
        task_id = record.get('task_id') or record.get('id') 
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        try:
            sel = run_phase1(record, z_threshold=Z_THRESHOLD, iter_cap=ITER_CAP, verbose=True)
            code = sel.get("code", "") if sel else ""
            # Save code
            output_file = out_dir / f"{task_id}.py"
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(code or "")

            # Persist result row
            results.append({
                "task_id": task_id,
                "iteration_used": sel.get("iteration"),
                "correct": sel.get("correctness"),
                "tests_passed": sel.get("passed"),
                "tests_failed": sel.get("failed"),
                "z_score": sel.get("z"),
                "p_value": sel.get("p"),
                "token_count": sel.get("token_count"),
                "green_count": sel.get("green_count"),
                "meets_z": sel.get("meets_z"),
                "stopped_early": sel.get("stopping_condition_met"),
                "output_file": str(output_file),
            })
        except Exception as e:
            print(f"[Phase1] ❌ Failure for {task_id}: {e}")
            results.append({
                "task_id": task_id,
                "error": str(e),
                "iteration_used": None,
                "correct": False,
                "tests_passed": 0,
                "tests_failed": 0,
                "z_score": float("nan"),
                "p_value": float("nan"),
                "token_count": 0,
                "green_count": 0,
                "meets_z": False,
                "stopped_early": False,
                "output_file": None,
            })
          
        print(f"\n{idx+1}/{len(df)} processed: {task_id}\n")
        
    results_df = pd.DataFrame(results)
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)    
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"Results saved to {RESULTS_CSV}")
    return results

In [ ]:
results = process_dataset(df, OUTPUT_DIR)